# Part 24: Model Context Protocol (MCP)

> MCP is an open standard for connecting AI agents to external tools and data sources. Build, consume, and compose MCP servers with any agent framework.

---


## 24.1 What is MCP?

MCP (Model Context Protocol) is like **USB for AI agents**:
- Standardized protocol for tool/resource connections
- **Servers** expose tools, resources, and prompts
- **Clients** (agents) discover and call those tools
- Works across frameworks: OpenAI Agents SDK, LangChain, Claude Desktop, etc.

### MCP Architecture

![MCP Architecture](https://raw.githubusercontent.com/sprashant433/GenAI/main/images/pattern_mcp_architecture.png)

- **Host** (your computer): contains multiple MCP Clients, one per server connection
- **Local MCP Servers**: connected via `stdio` (subprocess on your machine)
- **Remote MCP Servers**: connected over HTTP/SSE from a remote server
- Remote servers can themselves call external **APIs**

```
Agent (MCP Client)
    ↕ MCP Protocol (JSON-RPC over stdio/HTTP/SSE)
MCP Server
    ├── tools/     (callable functions)
    ├── resources/ (readable data)
    └── prompts/   (template prompts)
```

## 24.2 Using Existing MCP Servers

In [ ]:
# pip install openai-agents mcp mcp-server-fetch
from agents import Agent, Runner
from agents.mcp import MCPServerStdio
import asyncio

async def use_fetch_mcp():
    """Use the mcp-server-fetch to fetch web content."""
    async with MCPServerStdio(
        params={
            "command": "uvx",
            "args": ["mcp-server-fetch"]
        }
    ) as server:
        # List available tools
        tools = await server.list_tools()
        print("Available tools:", [t.name for t in tools])
        
        # Create agent with MCP server
        agent = Agent(
            name="Web Fetcher",
            instructions="Fetch and summarize web content.",
            mcp_servers=[server],
            model="gpt-4o-mini"
        )
        
        result = await Runner.run(agent, "Fetch https://example.com and summarize it")
        print(result.final_output)

# asyncio.run(use_fetch_mcp())


## 24.3 Playwright MCP (Browser Automation)

In [ ]:
async def use_playwright_mcp():
    """Browser automation via Playwright MCP."""
    async with MCPServerStdio(
        params={
            "command": "npx",
            "args": ["@playwright/mcp@latest"]
            # Requires: npm install -g @playwright/mcp
        }
    ) as playwright_server:
        tools = await playwright_server.list_tools()
        print("Browser tools:", [t.name for t in tools])
        # Tools include: navigate, click, type, screenshot, etc.
        
        agent = Agent(
            name="Browser Agent",
            instructions="""You control a web browser.
Use browser tools to navigate, interact, and extract information from websites.""",
            mcp_servers=[playwright_server],
            model="gpt-4o-mini"
        )
        
        result = await Runner.run(
            agent,
            "Go to wikipedia.org and find the article about Python programming language. Summarize the first paragraph."
        )
        return result.final_output

# asyncio.run(use_playwright_mcp())


## 24.4 Filesystem MCP

In [ ]:
async def use_filesystem_mcp(allowed_directory: str = "/tmp"):
    """File operations via the official MCP filesystem server."""
    async with MCPServerStdio(
        params={
            "command": "npx",
            "args": [
                "@modelcontextprotocol/server-filesystem",
                allowed_directory  # restrict to this directory
            ]
        }
    ) as fs_server:
        tools = await fs_server.list_tools()
        print("Filesystem tools:", [t.name for t in tools])
        # Tools: read_file, write_file, list_directory, create_directory, etc.
        
        agent = Agent(
            name="File Manager",
            instructions=f"Manage files in {allowed_directory}.",
            mcp_servers=[fs_server],
            model="gpt-4o-mini"
        )
        
        result = await Runner.run(
            agent,
            "Create a file called 'notes.txt' with a summary of MCP protocol."
        )
        return result.final_output


## 24.5 Building a Custom MCP Server

In [ ]:
# accounts_server.py — a custom MCP server

from mcp.server import FastMCP
from pydantic import BaseModel

# FastMCP is the easiest way to build MCP servers
mcp = FastMCP("Accounts Service")

# Mock database
ACCOUNTS = {
    "ACC001": {"name": "Alice Johnson", "balance": 1500.00, "currency": "USD"},
    "ACC002": {"name": "Bob Smith",     "balance": 342.50,  "currency": "USD"},
    "ACC003": {"name": "Carol White",   "balance": 8920.00, "currency": "EUR"},
}

@mcp.tool()
def get_account_balance(account_id: str) -> dict:
    """Get the balance for an account."""
    if account_id in ACCOUNTS:
        acc = ACCOUNTS[account_id]
        return {"account_id": account_id, "balance": acc["balance"], "currency": acc["currency"]}
    return {"error": f"Account {account_id} not found"}

@mcp.tool()
def transfer_funds(from_account: str, to_account: str, amount: float) -> dict:
    """Transfer funds between accounts."""
    if from_account not in ACCOUNTS or to_account not in ACCOUNTS:
        return {"error": "Invalid account"}
    if ACCOUNTS[from_account]["balance"] < amount:
        return {"error": "Insufficient funds"}
    
    ACCOUNTS[from_account]["balance"] -= amount
    ACCOUNTS[to_account]["balance"]   += amount
    return {"success": True, "transferred": amount}

@mcp.tool()
def list_accounts() -> list[dict]:
    """List all available accounts."""
    return [{"id": k, "name": v["name"]} for k, v in ACCOUNTS.items()]

# Resource: expose account data as a readable resource
@mcp.resource("accounts://all")
def get_all_accounts() -> str:
    """All account information as text."""
    lines = [f"{acc_id}: {info['name']} — {info['currency']}{info['balance']:.2f}"
             for acc_id, info in ACCOUNTS.items()]
    return "\n".join(lines)

# Run: uv run accounts_server.py
if __name__ == "__main__":
    mcp.run(transport="stdio")


## 24.6 Using a Custom MCP Server

In [ ]:
async def use_custom_accounts_server():
    """Connect to our custom accounts MCP server."""
    async with MCPServerStdio(
        params={
            "command": "uv",
            "args": ["run", "accounts_server.py"]
        }
    ) as server:
        # Discover tools
        tools = await server.list_tools()
        print("Accounts tools:", [t.name for t in tools])
        
        # Read resources
        resources = await server.list_resources()
        print("Resources:", [r.uri for r in resources])
        
        # Create agent
        finance_agent = Agent(
            name="Finance Agent",
            instructions="""You are a financial assistant.
Use the accounts tools to help users manage their accounts.
Always verify account IDs before transactions.""",
            mcp_servers=[server],
            model="gpt-4o-mini"
        )
        
        result = await Runner.run(
            finance_agent,
            "What is the balance of account ACC001? And list all accounts."
        )
        return result.final_output

# asyncio.run(use_custom_accounts_server())


## 24.7 MCP Client (Direct Protocol Access)

In [ ]:
# Direct MCP client without an agent framework
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def mcp_client_example():
    """Low-level MCP client usage."""
    server_params = StdioServerParameters(
        command="uv",
        args=["run", "accounts_server.py"]
    )
    
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            # Initialize
            await session.initialize()
            
            # List tools
            tools = await session.list_tools()
            print("Tools:", [t.name for t in tools.tools])
            
            # Call a tool directly
            result = await session.call_tool(
                "get_account_balance",
                arguments={"account_id": "ACC001"}
            )
            print("Balance:", result.content)
            
            # Read a resource
            resource = await session.read_resource("accounts://all")
            print("All accounts:\n", resource.contents[0].text)

# asyncio.run(mcp_client_example())


## 24.8 MCP with Multiple Servers

In [ ]:
async def multi_server_agent():
    """Agent using multiple MCP servers simultaneously."""
    async with (
        MCPServerStdio(params={"command": "uvx", "args": ["mcp-server-fetch"]}) as fetch_server,
        MCPServerStdio(params={"command": "uv",  "args": ["run", "accounts_server.py"]}) as accounts_server,
    ):
        # Agent has access to tools from BOTH servers
        super_agent = Agent(
            name="Multi-capability Agent",
            instructions="""You have access to:
1. Web fetching: fetch URLs, get current information
2. Account management: check balances, transfer funds

Use the right tool for each task.""",
            mcp_servers=[fetch_server, accounts_server],
            model="gpt-4o-mini"
        )
        
        result = await Runner.run(
            super_agent,
            "Check the balance of account ACC002 and fetch the current exchange rate for USD to EUR from https://example.com/rates"
        )
        return result.final_output


## 24.9 MCP Ecosystem

Popular MCP servers available:
| Server | Package | Capability |
|--------|---------|-----------|
| Fetch | `mcp-server-fetch` | Web fetching |
| Filesystem | `@modelcontextprotocol/server-filesystem` | File I/O |
| Browser | `@playwright/mcp` | Browser automation |
| GitHub | `@modelcontextprotocol/server-github` | GitHub API |
| Slack | `@modelcontextprotocol/server-slack` | Slack messaging |
| Postgres | `@modelcontextprotocol/server-postgres` | SQL queries |
| Puppeteer | `@modelcontextprotocol/server-puppeteer` | Headless browser |

Marketplaces: mcp.so, glama.ai, smithery.ai


## 24.10 Summary

| Concept | Implementation |
|---------|---------------|
| Use existing server | `MCPServerStdio(params={"command": "uvx", "args": [...]})` |
| Build server | `FastMCP("name")` + `@mcp.tool()` + `@mcp.resource()` |
| Run server | `mcp.run(transport="stdio")` |
| Direct client | `ClientSession` + `stdio_client` |
| Agent + MCP | `Agent(..., mcp_servers=[server])` |
| Multiple servers | `Agent(..., mcp_servers=[server1, server2])` |

---

**This completes the Generative AI Notebook Series!**

Return to [Index](index.ipynb) for the full overview.
